# Ejercicios loc/iloc

Crea un cuaderno nuevo, 04_loc_iloc.ipynb, y haz estos cinco pasos. En los que digo "predice", escribe tu respuesta en una celda Markdown antes de ejecutar:

Filtra los pingüinos con masa mayor de 5000 g y guárdalo en df_grandes. Imprime df_grandes.index.  
Predice y luego ejecuta: df_grandes.iloc[0].  
Predice y luego ejecuta: df_grandes.loc[0].  
Vuelve al DataFrame original y saca, en una sola instrucción con .loc, las columnas species y sex de los pingüinos de la isla Biscoe.  
Sobre el DataFrame original, ejecuta df.loc[0:5] y df.iloc[0:5]. Cuenta cuántas filas devuelve cada uno y dímelo.  

In [16]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("penguins")

## Ejercicio 1

Consideraciones de este ejercicio: al ejecutar el filtro de pingüinos de mayor masa de 5000 se observa que los índices de las filas empiezan con valores altos (221) en adelante. Eso nos dice que los datos se introdujeron en un orden específico (por islas, especie, etc.), es decir, el fichero viene ordenado y no barajado. Esa fue la pista que nos hizo sospechar y mirar más a fondo.

Al comprobar con value_counts() a qué especies corresponden nos da que las 61 filas son de una sola especie, Gentoo. Esto no se debe al orden del fichero: si alguien barajara las filas seguiríamos teniendo los mismos 61 Gentoo, porque la causa real es que los Gentoo son la especie más grande. El orden solo hizo visible el problema, no lo creó.

Por lo tanto el filtro que buscábamos de peso en la práctica se ha convertido en un filtro de especie. Ojo, df_grandes sí contiene a todos los pingüinos de más de 5000 g, no falta ninguno, el filtro hizo su trabajo. El problema es otro: peso y especie van tan pegados en estos datos que no se pueden distinguir. No tenemos ni un solo pingüino grande de otra especie con quien comparar, así que cualquier cosa que midamos sobre df_grandes no sabremos si se debe a que son grandes o a que son Gentoo.

Conclusión práctica: cada vez que filtremos por una variable numérica, conviene mirar con value_counts() cómo han quedado repartidas las columnas de categoría. Cuesta diez segundos y evita sacar conclusiones falsas que suenan razonables.

In [17]:
# Filtra los pingüinos con masa mayor de 5000 g y guárdalo en df_grandes. Imprime df_grandes.index.

df_grandes = df.loc[df["body_mass_g"] > 5000]
df_grandes.index
df_grandes["species"].value_counts()

species
Gentoo    61
Name: count, dtype: int64

In [18]:
#Consideración: cuando ponemos una etiqueta entre corchetes obtenemos un DataFrame (una tabla,
# aunque tenga solo una columna). Si lo ponemos sin corchetes obtenemos una Series (la columna suelta).
# El motivo es que los corchetes construyen una lista, y una lista siempre produce tabla, tenga
# un elemento o tenga veinte.


a = df.loc[df["body_mass_g"] > 5000, ["body_mass_g"]]   # DataFrame, shape (61, 1)
b = df.loc[df["body_mass_g"] > 5000, "body_mass_g"]     # Series, shape (61,)

#Esto es relevante porque al operar los resultados no son iguales:

valor_a = a.mean() #  ->  una Series de un elemento: .mean() da un resultado por cada 
valor_b = b.mean() #  -> y ese resultado lleva pegado el nombre de la columna un número normal: 5501.63

print(valor_a)
print(valor_b)

# Al intentar convertir a.mean() con float() el número sale bien, pero pandas lanza un FutureWarning
# avisando de que eso dejará de funcionar y dará TypeError en el futuro. Es un aviso, no un error:
# la celda se ejecuta igual y puede pasar desapercibido entre mucha salida.

# Regla: si voy a operar con números, pido Series (sin corchetes). Si voy a seguir tratándolo como
# tabla, pido DataFrame (con corchetes). Ante la duda, .shape lo aclara: un número entre paréntesis
# es Series, dos números es tabla.


body_mass_g    5501.639344
dtype: float64
5501.639344262295


# Ejercicio 2

Consideraciones:  

Cuando filtramos por posicion con "iloc" estamos filtrando por posición de la fila, lo que era una fila horizontal pasa a ser una  columna vertical, y los nombres de las columnas pasan a ser el índice.

Esta Series resultante del ejercicio  pasa a ser de tipo Object por lo que todo lo que sea agregar, comparar o promediar se hace por columnas ya que la Series entera lleva varios tipos de datos (int,str, float, etc.), sin embargo si la Serie resultante en su caso fuera del mismo tipo pasaria a ser de ese tipo de dato, no esta atado a ser Object siempre.

In [19]:
# Predice y luego ejecuta: df_grandes.iloc[0].

# La consulta mostrará la primera fila del dataset filtrado

df_grandes.iloc[0]


species              Gentoo
island               Biscoe
bill_length_mm         50.0
bill_depth_mm          16.3
flipper_length_mm     230.0
body_mass_g          5700.0
sex                    Male
Name: 221, dtype: object

In [20]:
print(type(df_grandes.iloc[0]["body_mass_g"]))
print(type(df_grandes["body_mass_g"].iloc[0]))

<class 'numpy.float64'>
<class 'numpy.float64'>


In [21]:
# df_grandes.iloc[0].mean()        # la fila entera, con textos dentro da fallo porque no puede promediar sobre un Object
df_grandes["body_mass_g"].mean()   # la columna, solo números promediará porqué es de tipo númerico

np.float64(5501.639344262295)

In [22]:
# Aqui podemos ver como si la fila lleva todos los datos del mismo tipo considerará el conjunto de ese dato no como Object

print(df_grandes.iloc[0].dtype)

df_grandes[["bill_length_mm", "bill_depth_mm", "body_mass_g"]].iloc[0].dtype

object


dtype('float64')

In [23]:
# Promedio sin sentido que promedia sobre magnitudes muy diferentes

df_grandes[["bill_length_mm", "bill_depth_mm", "body_mass_g"]].iloc[0].mean()

np.float64(1922.1000000000001)

# Ejercicio 3

In [24]:
# Predice y luego ejecuta: df_grandes.loc[0].

df_grandes.loc[0]  # --> KeyError, error de clave no encontrada


KeyError: 0

## Ejercicio 4

## Modificar un DataFrame: por qué .loc con coma es obligatorio

Consideración: los filtros en pandas devuelven una **copia**, no una ventana al original.
Esto se comprobó guardando un filtro en una variable:

    df_biscoe = df.loc[df["island"] == "Biscoe", ["species", "sex"]]
    df_biscoe["sex"] = "desconocido"

    df_biscoe -> todo "desconocido"
    df.loc[20, "sex"] -> sigue diciendo "Female"

Es decir, df_biscoe es una fotocopia hecha en el momento de ejecutar la línea, no una consulta
que se vuelva a lanzar. Escribir en la fotocopia no toca el original, y si el original cambiara
la fotocopia seguiría mostrando los datos de antes.

### El problema viene cuando lo que queremos es modificar el original

Estas dos líneas parecen equivalentes y no lo son:

    df[df["island"] == "Biscoe"]["sex"] = "desconocido"      # NO funciona
    df.loc[df["island"] == "Biscoe", "sex"] = "desconocido"  # SÍ funciona

La primera son dos operaciones separadas: primero el filtro crea una copia nueva, después se
escribe sobre esa copia. Como la copia no se guarda en ninguna variable, Python la borra y el
cambio se pierde. df queda intacto.

La segunda es una sola operación: se le dice a df directamente en qué filas y en qué columna
escribir. No se crea nada intermedio.

Comprobación:

    df = sns.load_dataset("penguins")

    df[df["island"] == "Biscoe"]["sex"] = "desconocido"
    print(df.loc[20, "sex"])    ->  Female        (no ha cambiado nada)

    df.loc[df["island"] == "Biscoe", "sex"] = "desconocido"
    print(df.loc[20, "sex"])    ->  desconocido   (ahora sí)

### Sobre el aviso

La primera línea lanza un SettingWithCopyWarning que dice literalmente que se está intentando
escribir sobre una copia y que se use .loc[row_indexer, col_indexer] en su lugar.

Pero es un aviso, no un error: la celda se ejecuta entera, sale el tick verde y el aviso aparece
al final de toda la salida. En un cuaderno largo pasa desapercibido con facilidad. Avisar no sirve
de nada si nadie mira, así que sigue contando como error silencioso en la práctica.

### Regla

Si voy a asignar, siempre .loc con coma. Nunca dos corchetes seguidos.
Esta es la razón real por la que .loc tiene esa sintaxis: no es comodidad para escribir menos,
es la única forma de modificar un subconjunto del DataFrame original.

In [ ]:
# Vuelve al DataFrame original y saca, en una sola instrucción con .loc, 
# las columnas species y sex de los pingüinos de la isla Biscoe. 

df_biscoe = df.loc[df["island"] == "Biscoe", ["species","sex"]]
print(df_biscoe)

    species          sex
20   Adelie  desconocido
21   Adelie  desconocido
22   Adelie  desconocido
23   Adelie  desconocido
24   Adelie  desconocido
..      ...          ...
339  Gentoo  desconocido
340  Gentoo  desconocido
341  Gentoo  desconocido
342  Gentoo  desconocido
343  Gentoo  desconocido

[168 rows x 2 columns]


In [ ]:
df_biscoe["sex"] = "desconocido"

print(df_biscoe["sex"].head())
print(df.loc[20, "sex"])

20    desconocido
21    desconocido
22    desconocido
23    desconocido
24    desconocido
Name: sex, dtype: object
desconocido


In [ ]:
df = sns.load_dataset("penguins")

df[df["island"] == "Biscoe"]["sex"] = "desconocido"
print(df.loc[20, "sex"])

df.loc[df["island"] == "Biscoe", "sex"] = "desconocido"
print(df.loc[20, "sex"])

Female
desconocido


/tmp/ipykernel_4363/2556998636.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[df["island"] == "Biscoe"]["sex"] = "desconocido"


## Ejercicio 5

In [ ]:
# Sobre el Data Frame original, ejecuta df.loc[0:5] y df.iloc[0:5]. 
# Cuenta cuántas filas devuelve cada uno y dímelo. 

df = sns.load_dataset("penguins")
df.loc[0:5] # Muestra 6 filas 
df.iloc[0:5] # Muestra 5 filas ya que iloc funciona por posiciones y como pas en  python excluye la última

